# Notebook 01 — Phase 1: AlexNet Training

Train the AlexNet CNN to classify images as **landslide** vs **non-landslide**.  
**Prerequisites**: Run notebook 00 first to set up the dataset.

Architecture: 5 Conv layers + 3 FC layers, 227×227 input, binary softmax output.

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

import torch
import config

print(f'Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

## 1. Build DataLoaders

In [ ]:
from phase1_alexnet.dataset import get_dataloaders

train_loader, test_loader = get_dataloaders(
    processed_dir=config.PROCESSED_DATA_DIR,
    batch_size=config.BATCH_SIZE,
    use_weighted_sampler=True,   # handles 3:1 class imbalance
)

print(f'Train batches: {len(train_loader)}')
print(f'Test  batches: {len(test_loader)}')

# Check a batch
imgs, labels = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}, labels: {labels.unique().tolist()}')

## 2. Model Architecture

In [ ]:
from phase1_alexnet.model import get_model

# Set pretrained=True to use ImageNet weights for faster convergence
model = get_model(pretrained=False, num_classes=config.NUM_CLASSES)

try:
    from torchinfo import summary
    summary(model, input_size=(1, 3, 227, 227))
except ImportError:
    print(model)

## 3. Verify Forward Pass

In [ ]:
import torch

dummy = torch.randn(4, 3, 227, 227)
out = model(dummy)
assert out.shape == (4, 2), f'Expected (4,2), got {out.shape}'
print(f'Forward pass OK — output shape: {out.shape}')

## 4. Train the Model

In [ ]:
from phase1_alexnet.train import run_training

# Reduce NUM_EPOCHS for quick testing; use 30 for full training
trained_model, history = run_training(
    num_epochs=config.NUM_EPOCHS,
    batch_size=config.BATCH_SIZE,
    learning_rate=config.LEARNING_RATE,
    pretrained=False,
    processed_dir=config.PROCESSED_DATA_DIR,
)

## 5. Plot Training History

In [ ]:
from utils.plot_utils import plot_training_history

plot_training_history(
    history,
    save_path=os.path.join(config.PLOTS_DIR, 'training_history.png')
)

## 6. Quick Validation Check

In [ ]:
print(f'Best training accuracy  : {max(history["train_acc"]):.4f}')
print(f'Best validation accuracy: {max(history["val_acc"]):.4f}')
print(f'Final train loss: {history["train_loss"][-1]:.4f}')
print(f'Final val   loss: {history["val_loss"][-1]:.4f}')

overfit_gap = max(history['train_acc']) - max(history['val_acc'])
if overfit_gap > 0.10:
    print(f'WARNING: Possible overfitting (gap={overfit_gap:.4f}). Consider more augmentation or dropout.')
else:
    print('Overfitting check: OK')